In [1]:
import array

from recbole.data.dataset.embeddings.creating_emb_file import embeddings


def readImageFeatures(path):
    with open(path, 'rb') as f:
        while True:
            asin = f.read(10)
            if asin == b'':  # koniec pliku
                break
            asin = asin.decode("utf-8")

            a = array.array('f')
            try:
                a.fromfile(f, 4096)
            except EOFError:
                break  # plik się skończył w trakcie

            yield asin.strip(), a.tolist()


In [1]:
import json

with open("/Users/jakubmalczak/UNI/INŻ/SequentialRecommendation/recbole/data/dataset/Amazon_Sports_and_Outdoors/item_reverse_mapping_Amazon_Sports_and_Outdoors.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [3]:
import numpy as np
import struct

file_path = 'image_features_Sports_and_Outdoors.b'
record_size = 10 + 4096 * 4

product_ids = []
features = []

with open(file_path, 'rb') as f:
    while True:
        bytes_chunk = f.read(record_size)
        if not bytes_chunk:
            break
        # ID produktu (10 bajtów → string)
        prod_id = bytes_chunk[:10].decode('utf-8')
        if prod_id not in data:
            continue
        product_ids.append(data[prod_id])
        # cechy obrazu (4096 floatów)
        feat = struct.unpack('f'*4096, bytes_chunk[10:])
        features.append(feat)

features = np.array(features, dtype=np.float32)
print(product_ids[:5])
print(features.shape)  # powinno być (liczba_produktów, 4096)

[2343, 1686, 1988, 2500, 2149]
(18177, 4096)


In [5]:
import pandas as pd
df = pd.read_csv('/Users/jakubmalczak/UNI/INŻ/SequentialRecommendation/recbole/data/dataset/Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.item', delimiter='\t')

df = df[['item_id:token']]
df.rename(columns={'item_id:token' : 'ent_id:token'}, inplace=True)
df.shape

(18357, 1)

In [7]:
df_features = pd.DataFrame({
    'ent_id:token': product_ids,
    'ent_emb:float_seq': [' '.join(map(str, vec)) for vec in features]
})

for x in features:
    assert len(x) == 4096

df_features.head(5)

,ent_id:token,ent_emb:float_seq
0,2343,0.0 0.0 1.4426 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0...
1,1686,0.0 0.0 1.2882 0.0 0.0 0.0 0.0 0.0 1.521 0.0 0...
2,1988,0.0 0.0 0.0 1.7559 0.0 0.0 0.6442 1.582 0.0 0....
3,2500,0.0 0.0 0.0 0.0 0.0 0.0 1.5838 0.0 0.0 0.0 0.0...
4,2149,0.0 0.0 0.0 0.0 0.0 0.1243 1.374 0.0 0.0 0.0 0...


In [8]:
df_merged = df.merge(df_features, on='ent_id:token', how='left')

In [11]:
nan_rows = df_merged[df_merged.isna().any(axis=1)]
print(nan_rows)

       ent_id:token ent_emb:float_seq
126             126               NaN
328             328               NaN
1253           1252               NaN
1908           1908               NaN
1971           1971               NaN
...             ...               ...
16900         16900               NaN
17437         17438               NaN
17908         17908               NaN
17909         17910               NaN
18226         18226               NaN

[180 rows x 2 columns]


In [12]:
embed_dim = 4096
zero_embedding = ' '.join(['0.0'] * embed_dim)

# Zamiana NaN na zero embedding
df_merged['ent_emb:float_seq'] = df_merged['ent_emb:float_seq'].fillna(zero_embedding)
nan_rows = df_merged[df_merged.isna().any(axis=1)]
print(nan_rows)

Empty DataFrame
Columns: [ent_id:token, ent_emb:float_seq]
Index: []


In [13]:
df_merged.to_csv('Amazon_Sports_and_Outdoors.image_original', sep="\t", index=False)

In [14]:
from sklearn.kernel_approximation import Nystroem

nys = Nystroem(kernel="cosine", n_components=128, random_state=42)

embeddings_128 = nys.fit_transform(features)

In [15]:
df_features['ent_emb:float_seq'] = [' '.join(map(str, vec)) for vec in embeddings_128]

In [18]:
df = df[['ent_id:token']]
df_merged = df.merge(df_features, on='ent_id:token', how='left')
nan_rows = df_merged[df_merged.isna().any(axis=1)]
print(nan_rows)

       ent_id:token ent_emb:float_seq
126             126               NaN
328             328               NaN
1253           1252               NaN
1908           1908               NaN
1971           1971               NaN
...             ...               ...
16900         16900               NaN
17437         17438               NaN
17908         17908               NaN
17909         17910               NaN
18226         18226               NaN

[180 rows x 2 columns]


In [19]:
embed_dim = 128
zero_embedding = ' '.join(['0.0'] * embed_dim)

# Zamiana NaN na zero embedding
df_merged['ent_emb:float_seq'] = df_merged['ent_emb:float_seq'].fillna(zero_embedding)
nan_rows = df_merged[df_merged.isna().any(axis=1)]
print(nan_rows)

Empty DataFrame
Columns: [ent_id:token, ent_emb:float_seq]
Index: []


In [20]:
df_merged.to_csv('Amazon_Sports_and_Outdoors.image', sep="\t", index=False)

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

orig_sim = cosine_similarity(features[:1000])
reduced_sim = cosine_similarity(embeddings_128[:1000])
correlation = np.corrcoef(orig_sim.flatten(), reduced_sim.flatten())[0, 1]

# ~0.9 is quite good
print("Similarity correlation:", correlation)

Similarity correlation: 0.9279410599732814
